**Imports and Dependencies**
Imports all required libraries including PyTorch, Hugging Face Transformers, Datasets, evaluation metrics, Weights & Biases logging, and utilities for running external COMET scoring.

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from datasets import Dataset, concatenate_datasets
import evaluate
import numpy as np
import os
import wandb
import subprocess
import tempfile
import json

**Training Configuration Class**
Central configuration object defining model, training hyperparameters, language codes, evaluation settings, and COMET parameters. Uses aggressive specialization settings (high LR + high dropout throughout).

In [ ]:
class TrainingConfig:
    model_name = "facebook/nllb-200-distilled-600M"
    task = "nepali_tamang_bidirectional_translation"
    learning_rate = 5e-4  # High LR for all epochs
    dropout = 0.3  # High dropout for all epochs
    batch_size = 4
    gradient_accumulation_steps = 4
    num_epochs = 5
    max_length = 128
    eval_steps = 500
    save_steps = 1000
    logging_steps = 500
    generation_max_length = 128
    generation_num_beams = 4
    fp16 = True
    nep_lang = "npi_Deva"
    hin_lang = "hin_Deva"
    metric_for_best_model = "eval_sacrebleu"
    warmup_steps = 500
    weight_decay = 0.01
    comet_model = "Unbabel/wmt22-comet-da"
    comet_batch_size = 4
    comet_use_cpu = False

config = TrainingConfig()

**Data Loading Utility**
Reads parallel text files (one sentence per line) and ensures they have identical line counts.

In [ ]:
def load_data_files(nep_file, hin_file):
    with open(nep_file, 'r', encoding='utf-8') as f:
        nep_texts = [line.strip() for line in f.readlines()]
    with open(hin_file, 'r', encoding='utf-8') as f:
        hin_texts = [line.strip() for line in f.readlines()]
    assert len(nep_texts) == len(hin_texts), "Files must have same number of lines"
    return nep_texts, hin_texts

**Dataset Creation (Bidirectional + Test Splits)**
Creates bidirectional training data (Nepali→Tamang + Tamang→Nepali) by duplicating and reversing pairs. Also prepares separate directional test sets.

In [ ]:
def create_datasets(train_nep, train_hin, test_nep, test_hin):
    train_nep_data, train_hin_data = load_data_files(train_nep, train_hin)
    test_nep_data, test_hin_data = load_data_files(test_nep, test_hin)

    train_forward = Dataset.from_dict({
        "source": train_nep_data,
        "target": train_hin_data,
        "src_lang": [config.nep_lang] * len(train_nep_data),
        "tgt_lang": [config.hin_lang] * len(train_nep_data)
    })

    train_backward = Dataset.from_dict({
        "source": train_hin_data,
        "target": train_nep_data,
        "src_lang": [config.hin_lang] * len(train_hin_data),
        "tgt_lang": [config.nep_lang] * len(train_hin_data)
    })

    train_combined = concatenate_datasets([train_forward, train_backward])

    test_nep2hin = Dataset.from_dict({
        "source": test_nep_data,
        "target": test_hin_data,
        "src_lang": [config.nep_lang] * len(test_nep_data),
        "tgt_lang": [config.hin_lang] * len(test_nep_data)
    })

    test_hin2nep = Dataset.from_dict({
        "source": test_hin_data,
        "target": test_nep_data,
        "src_lang": [config.hin_lang] * len(test_hin_data),
        "tgt_lang": [config.nep_lang] * len(test_hin_data)
    })

    return {
        "train": train_combined,
        "test_nep2hin": test_nep2hin,
        "test_hin2nep": test_hin2nep,
        "dataset_info": {
            "train_size": len(train_nep_data),
            "combined_train_size": len(train_nep_data) * 2,
            "test_size": len(test_nep_data),
            "total_size": len(train_nep_data) * 2 + len(test_nep_data)
        }
    }

**Preprocessing Function (Manual Tokenization with Language Tokens)**
Custom preprocessing that manually adds target language tokens at the beginning of decoder inputs (required by NLLB). Avoids automatic forced BOS token issues.

In [ ]:
def create_preprocess_function(tokenizer):
    def preprocess_data(examples):
        sources = examples["source"]
        targets = examples["target"]
        src_langs = examples["src_lang"]
        tgt_langs = examples["tgt_lang"]

        input_ids = []
        attention_masks = []
        labels = []

        for source, target, src_lang, tgt_lang in zip(sources, targets, src_langs, tgt_langs):
            tokenizer.src_lang = src_lang
            source_encoding = tokenizer(source, max_length=128, truncation=True, padding=False)

            target_tokens = tokenizer(
                target, max_length=126, truncation=True,
                padding=False, add_special_tokens=False
            )["input_ids"]

            tgt_lang_id = tokenizer.convert_tokens_to_ids(tgt_lang)
            target_ids = [tgt_lang_id] + target_tokens + [tokenizer.eos_token_id]

            input_ids.append(source_encoding["input_ids"])
            attention_masks.append(source_encoding["attention_mask"])
            labels.append(target_ids)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_masks,
            "labels": labels
        }
    return preprocess_data

**COMET Score Computation via External CLI Tool**
Runs the external comet-score CLI tool using temporary files and parses the system-level COMET score robustly across different output formats.


In [ ]:
def compute_comet_score(sources, predictions, references, model_name=None, batch_size=8, use_cpu=False):

    if model_name is None:
        model_name = config.comet_model

    try:
        # Create temporary files for sources, predictions, and references
        with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', suffix='.txt', delete=False) as src_f, \
             tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', suffix='.txt', delete=False) as hyp_f, \
             tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', suffix='.txt', delete=False) as ref_f:

            # Write data to temporary files
            src_f.write('\n'.join(sources))
            hyp_f.write('\n'.join(predictions))
            ref_f.write('\n'.join(references))

            src_file = src_f.name
            hyp_file = hyp_f.name
            ref_file = ref_f.name

        # Run comet-score command
        cmd = [
            'comet-score',
            '-s', src_file,
            '-t', hyp_file,
            '-r', ref_file,
            '--model', model_name,
            '--batch_size', str(batch_size),
            '--quiet',
            '--only_system'
        ]

        # Force CPU usage if specified or if we detect memory issues
        if use_cpu:
            cmd.extend(['--gpus', '0'])

        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            check=True
        )

        # Parse the output to get the score
        output = result.stdout.strip()

        # Try to parse as JSON first (newer versions output JSON)
        try:
            score_data = json.loads(output)
            if isinstance(score_data, dict) and 'system_score' in score_data:
                score = float(score_data['system_score'])
            else:
                score = float(output)
        except (json.JSONDecodeError, ValueError):
            # Try to extract score from text output
            try:
                score = float(output)
            except ValueError:
                # Look for score in output lines
                for line in output.split('\n'):
                    if 'score' in line.lower():
                        try:
                            score = float(line.split(':')[-1].strip())
                            break
                        except:
                            continue
                else:
                    score = 0.0

        # Clean up temporary files
        os.unlink(src_file)
        os.unlink(hyp_file)
        os.unlink(ref_file)


        return score

    except subprocess.CalledProcessError as e:
        print(f"Warning: comet-score command failed: {e.stderr}")
        # Clean up files if they exist
        try:
            os.unlink(src_file)
            os.unlink(hyp_file)
            os.unlink(ref_file)
        except:
            pass
        return 0.0
    except Exception as e:
        print(f"Warning: COMET computation failed: {str(e)}")
        return 0.0

**Metrics Computation (SacreBLEU, ChrF, METEOR + COMET)**
Computes multiple translation metrics during training/validation, including source-aware COMET via external tool.

In [ ]:
def create_compute_metrics_function(tokenizer):
    def compute_metrics(eval_preds):
        predictions, labels = eval_preds

        decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

        decoded_preds = [pred.strip() for pred in decoded_preds]
        decoded_labels = [label.strip() for label in decoded_labels]

        # Load basic metrics
        sacrebleu = evaluate.load("sacrebleu")
        chrf = evaluate.load("chrf")
        meteor = evaluate.load("meteor")

        # Compute basic metrics
        sacrebleu_result = sacrebleu.compute(predictions=decoded_preds, references=decoded_labels)
        chrf_result = chrf.compute(predictions=decoded_preds, references=decoded_labels, word_order=0)
        chrfpp_result = chrf.compute(predictions=decoded_preds, references=decoded_labels, word_order=2)
        meteor_result = meteor.compute(predictions=decoded_preds, references=decoded_labels)

        # Compute COMET score using CLI tool
        sources = [""] * len(decoded_preds)
        comet_score = compute_comet_score(
            sources, decoded_preds, decoded_labels,
            batch_size=config.comet_batch_size,
            use_cpu=config.comet_use_cpu
        )

        return {
            "sacrebleu": sacrebleu_result["score"],
            "chrf": chrf_result["score"],
            "chrfpp": chrfpp_result["score"],
            "meteor": meteor_result["meteor"],
            "comet": comet_score
        }
    return compute_metrics

**Custom Trainer — SpecializationTrainer (High LR + High Dropout)**
Overrides Hugging Face Trainer to force high learning rate and dropout in every epoch and implements pure specialization training without stabilization phase.

In [ ]:
class SpecializationTrainer(Seq2SeqTrainer):

    def __init__(self, *args, **kwargs):
        self.specialization_lr = kwargs.pop('specialization_lr', 5e-4)
        self.specialization_dropout = kwargs.pop('specialization_dropout', 0.3)
        super().__init__(*args, **kwargs)
        self.current_epoch = 0
        self.samples_per_epoch = None

    def on_train_begin(self, args, state, control, **kwargs):
        dataset_size = len(self.train_dataset)
        batch_size = args.per_device_train_batch_size * args.gradient_accumulation_steps
        if args.world_size > 1:
            batch_size *= args.world_size
        self.samples_per_epoch = dataset_size
        self.steps_per_epoch = dataset_size // batch_size

        print("\n" + "="*80)
        print("SPECIALIZATION TRAINING STRATEGY INITIALIZED")
        print("="*80)
        print(f"Dataset size: {dataset_size}")
        print(f"Steps per epoch: {self.steps_per_epoch}")
        print(f"All Epochs (1-{args.num_train_epochs}): SPECIALIZATION MODE")
        print(f"Learning Rate: {self.specialization_lr:.2e}")
        print(f"Dropout: {self.specialization_dropout}")
        print("Goal: Continuous aggressive adaptation for Nepali-Tamang translation")
        print("="*80 + "\n")

        # Set initial dropout
        self._update_model_dropout(self.specialization_dropout)

        super().on_train_begin(args, state, control, **kwargs)

    def on_epoch_begin(self, args, state, control, **kwargs):
        if hasattr(state, 'epoch'):
            current_epoch_number = int(state.epoch) + 1
        else:
            current_epoch_number = (state.global_step // self.steps_per_epoch) + 1

        self.current_epoch = current_epoch_number

        print("\n" + "="*80)
        print(f"SPECIALIZATION MODE - EPOCH {current_epoch_number}/{args.num_train_epochs}")
        print("="*80)
        print(f"Learning Rate: {self.specialization_lr:.2e}")
        print(f"Dropout: {self.specialization_dropout}")
        print("Status: Maintaining high learning rate for continuous specialization")
        print("="*80 + "\n")

        # Ensure learning rate and dropout are correct
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = self.specialization_lr

        self._update_model_dropout(self.specialization_dropout)

        wandb.log({
            f"config/epoch_{current_epoch_number}_lr": self.specialization_lr,
            f"config/epoch_{current_epoch_number}_dropout": self.specialization_dropout,
            "epoch": current_epoch_number,
            "phase": "SPECIALIZATION"
        })

    def on_epoch_end(self, args, state, control, **kwargs):
        completed_epoch = int(state.epoch) + 1
        print("\n" + "="*80)
        print(f"SPECIALIZATION EPOCH {completed_epoch} COMPLETED")
        print("="*80 + "\n")

    def _update_model_dropout(self, dropout_rate):
        for module in self.model.modules():
            if isinstance(module, torch.nn.Dropout):
                module.p = dropout_rate


**Trainer Factory with Specialization Settings**
Configures training arguments and instantiates the custom SpecializationTrainer.



In [ ]:
def create_specialization_trainer(model, tokenizer, train_dataset, compute_metrics):
    training_args = Seq2SeqTrainingArguments(
        output_dir="./nllb-5e4-do-0.3-5epochs",
        num_train_epochs=config.num_epochs,
        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        learning_rate=config.learning_rate,
        weight_decay=config.weight_decay,
        warmup_steps=config.warmup_steps,
        eval_strategy="no",
        save_steps=config.save_steps,
        logging_steps=config.logging_steps,
        predict_with_generate=True,
        generation_max_length=config.generation_max_length,
        generation_num_beams=config.generation_num_beams,
        load_best_model_at_end=False,
        remove_unused_columns=False,
        fp16=config.fp16,
        report_to="wandb",
        run_name="specialization-only-translation",
        logging_dir='./logs',
        save_total_limit=3,
    )

    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

    trainer = SpecializationTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=None,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        specialization_lr=config.learning_rate,
        specialization_dropout=config.dropout,
    )

    return trainer

**Final Test Evaluation (Both Directions + Averaging)**
Runs final evaluation on both translation directions using real source-aware COMET and logs detailed results.


In [ ]:
def evaluate_test_directions(trainer, datasets, preprocess_data):

    print("\n" + "="*80)
    print("EVALUATION PHASE - COMPUTING METRICS ON TEST SET")
    print("="*80 + "\n")

    # Get original datasets for COMET (we need source texts)
    test_nep2hin_original = datasets["test_nep2hin"]
    test_hin2nep_original = datasets["test_hin2nep"]

    test_nep2hin = test_nep2hin_original.map(
        preprocess_data, batched=True,
        remove_columns=test_nep2hin_original.column_names
    )

    print("Evaluating Nepali to Tamang direction...")
    nep2hin_results = trainer.evaluate(eval_dataset=test_nep2hin)

    # Compute COMET with actual source texts for final evaluation
    try:
        print("Computing COMET score with source texts for Nepali to Tamang...")
        predictions = trainer.predict(test_dataset=test_nep2hin)
        decoded_preds = trainer.tokenizer.batch_decode(predictions.predictions, skip_special_tokens=True)
        decoded_preds = [pred.strip() for pred in decoded_preds]

        sources = test_nep2hin_original["source"]
        references = test_nep2hin_original["target"]

        nep2hin_comet = compute_comet_score(
            sources, decoded_preds, references,
            batch_size=config.comet_batch_size,
            use_cpu=config.comet_use_cpu
        )
        nep2hin_results['eval_comet'] = nep2hin_comet
    except Exception as e:
        print(f"Warning: Could not compute COMET with source texts: {e}")

    test_hin2nep = test_hin2nep_original.map(
        preprocess_data, batched=True,
        remove_columns=test_hin2nep_original.column_names
    )

    print("Evaluating Tamang to Nepali direction...")
    hin2nep_results = trainer.evaluate(eval_dataset=test_hin2nep)

    # Compute COMET with actual source texts for final evaluation
    try:
        print("Computing COMET score with source texts for Tamang to Nepali...")
        predictions = trainer.predict(test_dataset=test_hin2nep)
        decoded_preds = trainer.tokenizer.batch_decode(predictions.predictions, skip_special_tokens=True)
        decoded_preds = [pred.strip() for pred in decoded_preds]

        sources = test_hin2nep_original["source"]
        references = test_hin2nep_original["target"]

        hin2nep_comet = compute_comet_score(
            sources, decoded_preds, references,
            batch_size=config.comet_batch_size,
            use_cpu=config.comet_use_cpu
        )
        hin2nep_results['eval_comet'] = hin2nep_comet
    except Exception as e:
        print(f"Warning: Could not compute COMET with source texts: {e}")

    avg_sacrebleu = (nep2hin_results['eval_sacrebleu'] + hin2nep_results['eval_sacrebleu']) / 2
    avg_chrf = (nep2hin_results['eval_chrf'] + hin2nep_results['eval_chrf']) / 2
    avg_chrfpp = (nep2hin_results['eval_chrfpp'] + hin2nep_results['eval_chrfpp']) / 2
    avg_meteor = (nep2hin_results['eval_meteor'] + hin2nep_results['eval_meteor']) / 2
    avg_comet = (nep2hin_results['eval_comet'] + hin2nep_results['eval_comet']) / 2

    print("\n" + "="*80)
    print("NEPALI TO TAMANG TRANSLATION RESULTS")
    print("="*80)
    print("SacreBLEU: %.4f" % nep2hin_results['eval_sacrebleu'])
    print("ChrF: %.4f" % nep2hin_results['eval_chrf'])
    print("ChrF++: %.4f" % nep2hin_results['eval_chrfpp'])
    print("METEOR: %.4f" % nep2hin_results['eval_meteor'])
    print("COMET: %.4f" % nep2hin_results['eval_comet'])

    print("\n" + "="*80)
    print("TAMANG TO NEPALI TRANSLATION RESULTS")
    print("="*80)
    print("SacreBLEU: %.4f" % hin2nep_results['eval_sacrebleu'])
    print("ChrF: %.4f" % hin2nep_results['eval_chrf'])
    print("ChrF++: %.4f" % hin2nep_results['eval_chrfpp'])
    print("METEOR: %.4f" % hin2nep_results['eval_meteor'])
    print("COMET: %.4f" % hin2nep_results['eval_comet'])

    print("\n" + "="*80)
    print("AVERAGE RESULTS ACROSS BOTH DIRECTIONS")
    print("="*80)
    print("Average SacreBLEU: %.4f" % avg_sacrebleu)
    print("Average ChrF: %.4f" % avg_chrf)
    print("Average ChrF++: %.4f" % avg_chrfpp)
    print("Average METEOR: %.4f" % avg_meteor)
    print("Average COMET: %.4f" % avg_comet)
    print("="*80 + "\n")

    wandb.log({
        "test/nep2hin_sacrebleu": nep2hin_results['eval_sacrebleu'],
        "test/nep2hin_chrf": nep2hin_results['eval_chrf'],
        "test/nep2hin_chrfpp": nep2hin_results['eval_chrfpp'],
        "test/nep2hin_meteor": nep2hin_results['eval_meteor'],
        "test/nep2hin_comet": nep2hin_results['eval_comet'],
        "test/hin2nep_sacrebleu": hin2nep_results['eval_sacrebleu'],
        "test/hin2nep_chrf": hin2nep_results['eval_chrf'],
        "test/hin2nep_chrfpp": hin2nep_results['eval_chrfpp'],
        "test/hin2nep_meteor": hin2nep_results['eval_meteor'],
        "test/hin2nep_comet": hin2nep_results['eval_comet'],
        "test/avg_sacrebleu": avg_sacrebleu,
        "test/avg_chrf": avg_chrf,
        "test/avg_chrfpp": avg_chrfpp,
        "test/avg_meteor": avg_meteor,
        "test/avg_comet": avg_comet,
    })

    return {
        "nep2hin": {
            "sacrebleu": nep2hin_results['eval_sacrebleu'],
            "chrf": nep2hin_results['eval_chrf'],
            "chrfpp": nep2hin_results['eval_chrfpp'],
            "meteor": nep2hin_results['eval_meteor'],
            "comet": nep2hin_results['eval_comet']
        },
        "hin2nep": {
            "sacrebleu": hin2nep_results['eval_sacrebleu'],
            "chrf": hin2nep_results['eval_chrf'],
            "chrfpp": hin2nep_results['eval_chrfpp'],
            "meteor": hin2nep_results['eval_meteor'],
            "comet": hin2nep_results['eval_comet']
        },
        "averages": {
            "sacrebleu": avg_sacrebleu,
            "chrf": avg_chrf,
            "chrfpp": avg_chrfpp,
            "meteor": avg_meteor,
            "comet": avg_comet
        }
    }


**Main Training Orchestration Function**
Complete end-to-end pipeline: login to wandb → load model → prepare data → train with specialization strategy → evaluate both directions.

In [ ]:
def run_specialization_training(train_nep_file, train_hin_file,
                                 test_nep_file, test_hin_file):

    os.environ["WANDB_API_KEY"] = "enter-your-api-key"
    wandb.login()

    print("\n" + "="*80)
    print("STARTING SPECIALIZATION-ONLY TRAINING PIPELINE")
    print("="*80 + "\n")

    wandb_config = {
        "model_name": config.model_name,
        "task": config.task,
        "learning_rate": config.learning_rate,
        "dropout": config.dropout,
        "batch_size": config.batch_size,
        "gradient_accumulation_steps": config.gradient_accumulation_steps,
        "num_epochs": config.num_epochs,
        "max_length": config.max_length,
        "eval_steps": config.eval_steps,
        "save_steps": config.save_steps,
        "logging_steps": config.logging_steps,
        "generation_max_length": config.generation_max_length,
        "generation_num_beams": config.generation_num_beams,
        "fp16": config.fp16,
        "nep_lang": config.nep_lang,
        "hin_lang": config.hin_lang,
        "metric_for_best_model": config.metric_for_best_model,
        "warmup_steps": config.warmup_steps,
        "weight_decay": config.weight_decay,
        "comet_model": config.comet_model,
        "training_strategy": "specialization_only"
    }

    run = wandb.init(
        project="nepali-tamang-translation",
        name="nllb-5e4-0.3-5epochs",
        config=wandb_config,
        tags=["nepali", "tamang", "translation", "specialization-only", "nllb"]
    )

    print("Loading model and tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(config.model_name)

    preprocess_data = create_preprocess_function(tokenizer)
    compute_metrics = create_compute_metrics_function(tokenizer)

    print("Creating datasets...")
    datasets = create_datasets(train_nep_file, train_hin_file,
                              test_nep_file, test_hin_file)

    wandb.log(datasets["dataset_info"])

    print("Preprocessing training data...")
    train_dataset = datasets["train"].map(
        preprocess_data, batched=True,
        remove_columns=datasets["train"].column_names
    )

    trainer = create_specialization_trainer(model, tokenizer, train_dataset, compute_metrics)

    print("\nStarting training...\n")
    trainer.train()

    results = evaluate_test_directions(trainer, datasets, preprocess_data)

    return trainer, results

**Entry Point**
Executes the full training and evaluation when script is run directly.

In [ ]:
if __name__ == "__main__":
    trainer, results = run_specialization_training(
        train_nep_file="/path/ne-train.txt",
        train_hin_file="/path/tmg-train.txt",
        test_nep_file="/path/ne-test.txt",
        test_hin_file="/path/tmg-test.txt"
    )